<a href="https://colab.research.google.com/github/awagler2/NODE/blob/main/tensorflow_encodings.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import OneHotEncoder

dat=pd.read_csv('year_inst.csv')
print(dat.head())

In [ ]:
import pandas as pd

# Find columns in 'dat' DataFrame containing 'hisp' (case-insensitive)
hisp_columns = [col for col in dat.columns if 'hisp' in col.lower()]

# Print the found columns
print("Columns containing 'hisp':", hisp_columns)

In [ ]:
categorical_features = dat.select_dtypes(include=['object', 'category'])
numerical_features = dat.select_dtypes(include=np.number)
print("All Columns with Index:")
for i, col in enumerate(dat.columns):
    print(f"{i}: {col}")
#print("\nCategorical Features:", categorical_features.columns)
#print("Numerical Features:", numerical_features.columns)

In [ ]:
columns_to_select = ['ADM_RATE','UGDS_WHITE','UGDS_BLACK','UGDS_ASIAN','UGDS_HISP','PCTPELL','UG25ABV','FEMALE','FIRST_GEN','POVERTY_RATE','MEDIAN_HH_INC','PCT_GRAD_PROF']
df_selected = dat[columns_to_select]
print(df_selected.head())

In [ ]:
print(dat.shape)

df_onehot = dat.iloc[:,[14,17,26,46,47,48,53,70,74,75,79,80,81,85,86,87,236,243,244,246,247] + list(range(270,274)) + [290,349,501,502,503,504,505]]
df_onehot['HSRU'] = dat['HSRU'].apply(lambda x: 'False' if x == 0 else 'True')

df_cont= dat.iloc[:,[73,78,84,90,92,93,95,96,117,121,122,130,137,145,173] + list(range(392,400)) + list(range(449,453)) + [468,471,493,494,499]]

# Combine the 'cc_basic' columns, prioritizing the most recent year
df_onehot['cc_basic_combined'] = dat['cc_basic_2021'].fillna(dat['cc_basic_2018']).fillna(dat['cc_basic_2015'])

# Combine the 'cc_undergrad' columns
dat['cc_undergrad_combined'] = dat['cc_undergrad_2021'].fillna(dat['cc_undergrad_2018']).fillna(dat['cc_undergrad_2015'])
df_onehot['cc_enroll_combined'] = dat['cc_enroll_2021'].fillna(dat['cc_enroll_2018']).fillna(dat['cc_enroll_2015'])
df_onehot['cc_basic_combined'] = dat['cc_basic_2021'].fillna(dat['cc_basic_2018']).fillna(dat['cc_basic_2015'])
df_onehot['cc_size_setting_combined'] = dat['cc_size_setting_2021'].fillna(dat['cc_size_setting_2018']).fillna(dat['cc_size_setting_2015'])
df_cont['completion_rate_combined'] = dat['completion_rate_100pct'].fillna(dat['completion_rate_150pct']).fillna(dat['completion_rate_200pct'])
df_cont['completers_combined'] = dat['completers_100pct'].fillna(dat['completers_150pct']).fillna(dat['completers_200pct'])
df_cont['gradrates_completers_combined'] = dat['gradrates_completers_100pct'].fillna(dat['gradrates_completers_150pct'])
df_cont['cohort_adj_combined'] = dat['cohort_adj_200pct'].fillna(dat['gradrates_completers_150pct'])

# Drop the original 'cc_basic' columns
df_cont = df_cont.drop(columns=['cc_undergrad_2015', 'cc_undergrad_2018', 'cc_undergrad_2021', 'completion_rate_100pct','completion_rate_150pct','completion_rate_200pct','completers_100pct','completers_150pct','completers_200pct','gradrates_completers_100pct','gradrates_completers_150pct','cohort_adj_150pct','cohort_adj_200pct'])
df_onehot=df_onehot.drop(columns=['cc_basic_2015','cc_basic_2018','cc_basic_2021','cc_size_setting_2015','cc_size_setting_2018','cc_size_setting_2021','cc_enroll_2015','cc_enroll_2018','cc_enroll_2021'])

df_cont = pd.concat([df_cont, df_selected], axis=1)

print(df_onehot.columns)
#print(df_cont.columns)

In [ ]:
# Drop columns with all missing values
df_cont_cleaned = df_cont.dropna(axis=1, how='all')

df_cont_centered = (df_cont_cleaned - df_cont_cleaned.mean())/df_cont_cleaned.std()

print(df_cont_centered.mean())

In [ ]:
from sklearn.preprocessing import OneHotEncoder
import pandas as pd
import numpy as np

# Convert all columns in df_onehot to string type
for col in df_onehot.columns:
    df_onehot[col] = df_onehot[col].astype(str)

# Identify non-numeric columns in df_onehot (after conversion)
non_numeric_cols = df_onehot.select_dtypes(exclude=np.number).columns
print("Non-numeric columns in df_onehot:", non_numeric_cols)

if len(non_numeric_cols) > 0:
    # Initialize OneHotEncoder
    # handle_unknown='ignore' is useful for handling categories not seen during fit
    # sparse_output=False returns a dense numpy array
    encoder = OneHotEncoder(handle_unknown='ignore', sparse_output=False)

    # Fit and transform the non-numeric columns
    onehot_encoded_data = encoder.fit_transform(df_onehot[non_numeric_cols])

    # Create a DataFrame from the one-hot encoded data
    # Get feature names for the new columns
    feature_names = encoder.get_feature_names_out(non_numeric_cols)
    onehot_encoded_df = pd.DataFrame(onehot_encoded_data, columns=feature_names, index=df_onehot.index)

    # Drop the original non-numeric columns from df_onehot
    df_onehot = df_onehot.drop(columns=non_numeric_cols)

    # Concatenate the original df_onehot (now only numeric) with the one-hot encoded DataFrame
    df_onehot = pd.concat([df_onehot, onehot_encoded_df], axis=1)

    print("df_onehot after one-hot encoding non-numeric columns:")
    display(df_onehot.head())
    print("Shape of df_onehot after one-hot encoding:", df_onehot.shape)
else:
    print("No non-numeric columns found in df_onehot.")

In [ ]:
from sklearn.preprocessing import OneHotEncoder
import pandas as pd
import numpy as np

# Identify non-numeric columns in df_onehot
non_numeric_cols = df_onehot.select_dtypes(exclude=np.number).columns
print("Non-numeric columns in df_onehot:", non_numeric_cols)

if len(non_numeric_cols) > 0:
    # Initialize OneHotEncoder
    # handle_unknown='ignore' is useful for handling categories not seen during fit
    # sparse_output=False returns a dense numpy array
    encoder = OneHotEncoder(handle_unknown='ignore', sparse_output=False)

    # Fit and transform the non-numeric columns
    onehot_encoded_data = encoder.fit_transform(df_onehot[non_numeric_cols])

    # Create a DataFrame from the one-hot encoded data
    # Get feature names for the new columns
    feature_names = encoder.get_feature_names_out(non_numeric_cols)
    onehot_encoded_df = pd.DataFrame(onehot_encoded_data, columns=feature_names, index=df_onehot.index)

    # Drop the original non-numeric columns from df_onehot
    df_onehot = df_onehot.drop(columns=non_numeric_cols)

    # Concatenate the original df_onehot (now only numeric) with the one-hot encoded DataFrame
    df_onehot = pd.concat([df_onehot, onehot_encoded_df], axis=1)

    print("df_onehot after one-hot encoding non-numeric columns:")
    display(df_onehot.head())
    print("Shape of df_onehot after one-hot encoding:", df_onehot.shape)
else:
    print("No non-numeric columns found in df_onehot.")

In [ ]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
import numpy as np
from sklearn.preprocessing import MinMaxScaler
from sklearn.impute import SimpleImputer

# Assuming df_onehot is a pandas DataFrame or numpy array
# Make sure df_onehot only contains numerical data types before building the model
df_onehot_numeric = df_onehot.select_dtypes(include=np.number).copy()
input_dim = df_onehot_numeric.shape[1]  # Number of features after one-hot encoding
encoding_dim = 10  # Desired reduced dimension

autoencoder = Sequential()
autoencoder.add(Dense(units=encoding_dim, activation='relu', input_shape=(input_dim,)))
autoencoder.add(Dense(units=input_dim, activation='sigmoid'))

# Compile the autoencoder
autoencoder.compile(optimizer='adam', loss='binary_crossentropy')

# Display model summary
autoencoder.summary()

# Fit the autoencoder
autoencoder.fit(df_onehot_numeric, df_onehot_numeric, epochs=50, batch_size=256)

In [ ]:
# Get the encoded features (latent representation) from the encoder layer
encoder_model = Sequential()
encoder_model.add(autoencoder.layers[0]) # Add the first layer (encoder) of the trained autoencoder

encoded_features = encoder_model.predict(df_onehot_numeric)
# Display the shape of the encoded features
print("Shape of encoded features:", encoded_features.shape)


In [ ]:
from sklearn.decomposition import PCA
from sklearn.impute import SimpleImputer
import pandas as pd
import numpy as np

# Impute missing values in df_cont_centered
imputer = SimpleImputer(missing_values=np.nan, strategy='mean')
df_cont_imputed = imputer.fit_transform(df_cont_centered)

# Apply PCA
pca = PCA(n_components=10) # You can adjust the number of components
df_pca = pca.fit_transform(df_cont_imputed)

# Convert PCA scores to a DataFrame
df_pca = pd.DataFrame(df_pca, columns=[f'pca_{i+1}' for i in range(df_pca.shape[1])])

# Display the head of the PCA scores
display(df_pca.head())

In [ ]:
encoded_features_df = pd.DataFrame(encoded_features)
encoded_features_df=encoded_features_df-encoded_features_df.mean()
encoded_features_df=encoded_features_df/encoded_features_df.std()



encoded_features_df = pd.concat([dat,encoded_features_df, df_pca],axis=1)

print(encoded_features_df.head())

print(encoded_features_df.shape)
encoded_features_df.to_csv('encoded_features.csv', index=False)

In [ ]:


# Select the last 25 columns and display the head
print(encoded_features_df.iloc[:, -20:].head())

